# Week 01 — Python for Machine Learning & Data Preprocessing

AI Pioneers Machine Learning Internship. This notebook prepares the UCI Bank Marketing dataset. When the UCI CSV is unavailable, a clearly labelled synthetic fixture is used only for a smoke test.

Official source: https://archive.ics.uci.edu/dataset/222/bank+marketing
Dataset DOI: https://doi.org/10.24432/C5K306

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebooks': PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))
from src.data_loader import download_and_extract, load_bank_marketing
from src.preprocessing import clean_data, outlier_summary, run_pipeline
print('NumPy', np.__version__, '| Pandas', pd.__version__)

## 1. Load the dataset
The UCI file is semicolon-delimited. The notebook tries to download the official file; if the download is unavailable, it uses the tiny synthetic offline fixture.

In [ ]:
official_path = PROJECT_DIR / 'data/raw/bank-additional-full.csv'
sample_path = PROJECT_DIR / 'data/sample/bank_marketing_demo_synthetic.csv'
if official_path.exists():
    data_path, source_label = official_path, 'UCI official dataset'
else:
    try:
        data_path = download_and_extract(PROJECT_DIR / 'data/raw')
        source_label = 'UCI official dataset (downloaded)'
    except Exception as exc:
        data_path, source_label = sample_path, 'SYNTHETIC DEMO FIXTURE — NOT UCI DATA'
        print('Official download unavailable:', exc)
raw_df = load_bank_marketing(data_path)
print(source_label, raw_df.shape)
display(raw_df.head())

## 2. Inspect with Pandas and NumPy
Review row/column counts, data types, missing values, duplicates, descriptive statistics, and selected numerical summaries.

In [ ]:
display(raw_df.dtypes.rename('dtype').to_frame())
display(raw_df.isna().sum().rename('missing').to_frame())
print('Exact duplicates:', int(raw_df.duplicated().sum()))
display(raw_df.describe(include='all').T)
ages = raw_df['age'].dropna().to_numpy(dtype=float)
print('Mean age:', round(float(np.mean(ages)), 2), 'Median:', float(np.median(ages)))
print('Vectorised age + 1 example:', (ages + 1)[:5])

## 3. Clean data
Trim text fields, convert blanks to missing values, remove exact duplicates, median-impute numeric nulls, and fill missing categories with `unknown`. Existing `unknown` category values are retained.

In [ ]:
cleaned_df, cleaning_summary = clean_data(raw_df)
print(json.dumps(cleaning_summary, indent=2, default=str))
print('Missing after cleaning:', int(cleaned_df.isna().sum().sum()))
display(cleaned_df.head())
display(outlier_summary(cleaned_df))

## 4. Exploratory analysis
Review the target response distribution and customer age. IQR outliers are reported, not automatically removed, because unusual values may be valid observations.

In [ ]:
display(cleaned_df['y'].value_counts().rename('records').to_frame())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cleaned_df['age'].plot(kind='hist', bins=20, ax=axes[0], edgecolor='white')
axes[0].set_title('Customer age distribution')
cleaned_df['y'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Term-deposit subscription response')
plt.tight_layout(); plt.show()

## 5. Feature selection, encoding, and scaling
The target `y` is separated. `duration` remains in the cleaned EDA table but is excluded from pre-call features because call duration is only known after the call, which can cause target leakage. Categorical variables are one-hot encoded and numeric variables standardized.

In [ ]:
output_dir = PROJECT_DIR / 'outputs' / ('full_dataset' if source_label.startswith('UCI') else 'demo')
summary = run_pipeline(raw_df, output_dir)
manifest = json.loads((output_dir / 'model_ready/preprocessing_manifest.json').read_text())
print('Output directory:', output_dir)
print('Train / test rows:', manifest['train_rows'], '/', manifest['test_rows'])
print('Encoded feature count:', manifest['encoded_feature_count'])
print('Excluded features:', manifest['excluded_from_features'])
print('Transform fit policy:', manifest['fit_policy'])

## 6. Conclusion
This practical demonstrates data loading, inspection, missing-value handling, duplicate removal, EDA, feature selection, categorical encoding, scaling, and a stratified train/test split. The transformation pipeline is fitted on the training partition only. No classifier is trained in this task. Any results generated from the fallback fixture are synthetic demonstration results, not results from the UCI dataset.